In [1]:
!pip install geopandas networkx numpy matplotlib pandas shapely

In [18]:
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point, LineString
from collections import defaultdict
from math import atan2, degrees, sqrt
import json

def extract_nodes_and_calculate_angles(gdf_lignes, angle_threshold=15.0):
    """
    Extrait les nœuds du réseau et calcule les angles de changement de direction.

    Args:
        gdf_lignes (GeoDataFrame): GeoDataFrame contenant les lignes électriques
        angle_threshold (float): Seuil en degrés pour considérer un angle comme significatif

    Returns:
        tuple: (df_nodes, df_angles) - DataFrames contenant les nœuds et les angles calculés
    """

    # =============================================
    # 1. EXTRACTION DES NŒUDS (POINTS DE CONNEXION)
    # =============================================

    # Dictionnaire pour stocker les nœuds et les lignes qui y sont connectées
    node_connections = defaultdict(list)
    node_coords = []  # Liste des coordonnées uniques des nœuds
    node_ids = {}     # Mapping coordonnées -> ID

    # Extraire tous les points de début et fin de chaque ligne
    for idx, row in gdf_lignes.iterrows():
        coords = list(row.geometry.coords)

        # Point de début
        start_coords = coords[0]
        if start_coords not in node_ids:
            node_ids[start_coords] = len(node_ids)
            node_coords.append(start_coords)
        node_connections[node_ids[start_coords]].append(idx)

        # Point de fin
        end_coords = coords[-1]
        if end_coords not in node_ids:
            node_ids[end_coords] = len(node_ids)
            node_coords.append(end_coords)
        node_connections[node_ids[end_coords]].append(idx)

    # Créer le DataFrame des nœuds
    df_nodes = pd.DataFrame({
        'node_id': list(node_ids.values()),
        'coords': list(node_ids.keys()),
        'ligne_ids': [node_connections[nid] for nid in node_connections.keys()],
        'degree': [len(node_connections[nid]) for nid in node_connections.keys()]
    })

    # =============================================
    # 2. CALCUL DES ANGLES POUR CHAQUE NŒUD
    # =============================================

    df_angles = []

    for _, row in df_nodes.iterrows():
        node_id = row['node_id']
        coords = row['coords']
        ligne_ids = row['ligne_ids']
        degree = row['degree']

        # Cas spécial: nœud terminal (début ou fin de ligne)
        if degree == 1:
            df_angles.append({
                'node_id': node_id,
                'coords': coords,
                'angle': None,
                'type': 'terminal',
                'degree': degree
            })
            continue

        # Trouver les lignes connectées au nœud
        connected_lines = []
        for line_id in ligne_ids:
            line_coords = list(gdf_lignes.iloc[line_id].geometry.coords)

            # Déterminer si le nœud est le début ou la fin de la ligne
            if line_coords[0] == coords:
                # Nœud est le début de la ligne
                if len(line_coords) > 1:
                    p1 = line_coords[1]
                    p1_prev = coords
                else:
                    p1 = coords
                    p1_prev = coords
            else:
                # Nœud est la fin de la ligne
                if len(line_coords) > 1:
                    p1_prev = line_coords[-2]
                    p1 = coords
                else:
                    p1_prev = coords
                    p1 = coords

            connected_lines.append((p1_prev, p1, line_id))

        # Calculer l'angle entre les lignes connectées
        if degree == 2:
            # Cas de 2 lignes connectées
            (p1_prev, p1, line1_id), (p2_prev, p2, line2_id) = connected_lines

            # Créer des vecteurs à partir du nœud
            v1 = np.array([p1[0] - coords[0], p1[1] - coords[1]])
            v2 = np.array([p2[0] - coords[0], p2[1] - coords[1]])

            # Calculer l'angle entre les deux vecteurs
            dot_product = np.dot(v1, v2)
            magnitude_v1 = np.sqrt(v1[0]**2 + v1[1]**2)
            magnitude_v2 = np.sqrt(v2[0]**2 + v2[1]**2)

            # Éviter la division par zéro
            if magnitude_v1 > 0 and magnitude_v2 > 0:
                cos_angle = dot_product / (magnitude_v1 * magnitude_v2)
                # Limiter la valeur pour éviter les erreurs d'arrondi
                cos_angle = np.clip(cos_angle, -1.0, 1.0)
                angle_rad = np.arccos(cos_angle)
                angle_deg = np.degrees(angle_rad)
            else:
                angle_deg = 0

            # Classifier le type de support
            if angle_deg > angle_threshold:
                support_type = 'angle'
            else:
                support_type = 'alignement'

            df_angles.append({
                'node_id': node_id,
                'coords': coords,
                'angle': angle_deg,
                'type': support_type,
                'degree': degree,
                'ligne_ids': ligne_ids
            })

        elif degree >= 3:
            # Cas d'étoilement (3+ lignes connectées)
            df_angles.append({
                'node_id': node_id,
                'coords': coords,
                'angle': None,
                'type': 'etoilement',
                'degree': degree,
                'ligne_ids': ligne_ids
            })

    # Créer le DataFrame des angles
    df_angles_final = pd.DataFrame(df_angles)

    return df_nodes, df_angles_final

# =============================================
# FONCTION POUR CHARGER LE GEOJSON CORRECTEMENT
# =============================================

def load_geojson(file_path):
    """
    Charge le fichier GeoJSON et crée un GeoDataFrame correctement structuré.

    Args:
        file_path (str): Chemin vers le fichier GeoJSON

    Returns:
        GeoDataFrame: Contenant les lignes électriques
    """
    with open(file_path, 'r', encoding='utf-8') as f:
        data = json.load(f)

    features = []
    for feature in data['features']:
        props = feature['properties']
        geom = feature['geometry']

        # Convertir les coordonnées en LineString
        if geom['type'] == 'MultiLineString':
            coords = geom['coordinates'][0]  # Prendre la première ligne
        else:
            coords = geom['coordinates']

        # Créer l'objet géométrique
        line_string = LineString(coords)

        features.append({
            'id': props.get('id', ''),
            'TFO': props.get('TFO', ''),
            'Type': props.get('Type', ''),
            'Type_2': props.get('Type_2', ''),
            'Longueur': props.get('Longueur', 0),
            'geometry': line_string  # Assigner la géométrie correctement
        })

    # Créer le GeoDataFrame avec la colonne geometry
    gdf = gpd.GeoDataFrame(features, geometry='geometry', crs="EPSG:4326")
    return gdf

# =============================================
# EXEMPLE D'UTILISATION
# =============================================

try:
    # Charger les données
    gdf_lignes = load_geojson("LigneBT_GrandLahou_Ext_Dens_geojson.geojson")
    print("✅ Données chargées avec succès!")
    print(f"📊 Nombre total de lignes : {len(gdf_lignes)}")
    print(f"📍 Système de coordonnées : {gdf_lignes.crs}")

    # Extraire les nœuds et calculer les angles
    df_nodes, df_angles = extract_nodes_and_calculate_angles(gdf_lignes, angle_threshold=15.0)

    print(f"\n📊 Résultats de l'extraction :")
    print(f"📍 Nombre total de nœuds: {len(df_nodes)}")
    print(f"📍 Nombre de nœuds avec angles calculés: {len(df_angles[df_angles['angle'].notna()])}")

    # Afficher un aperçu des résultats
    print("\n📋 Aperçu des nœuds extraits (5 premiers) :")
    print(df_nodes.head())

    print("\n📋 Aperçu des angles calculés (5 premiers) :")
    print(df_angles.head())

    # Statistiques
    print("\n📊 Statistiques :")
    print(f"📈 Répartition par type de support :")
    print(df_angles['type'].value_counts())

    if 'angle' in df_angles.columns and df_angles['angle'].notna().any():
        print(f"\n📐 Statistiques sur les angles :")
        print(f"- Angle moyen: {df_angles['angle'].mean():.2f}°")
        print(f"- Angle minimum: {df_angles['angle'].min():.2f}°")
        print(f"- Angle maximum: {df_angles['angle'].max():.2f}°")

    # Export des résultats (optionnel)
    df_nodes.to_csv('noeuds_classifies.csv', index=False)
    df_angles.to_csv('angles_calcules.csv', index=False)
    print("\n💾 Résultats exportés dans 'noeuds_classifies.csv' et 'angles_calcules.csv'")

except Exception as e:
    print(f"❌ Erreur lors du traitement: {str(e)}")
    import traceback
    traceback.print_exc()


✅ Données chargées avec succès!
📊 Nombre total de lignes : 120
📍 Système de coordonnées : EPSG:4326

📊 Résultats de l'extraction :
📍 Nombre total de nœuds: 235
📍 Nombre de nœuds avec angles calculés: 5

📋 Aperçu des nœuds extraits (5 premiers) :
   node_id                                   coords ligne_ids  degree
0        0  (-4.997630958956445, 5.245875938860019)       [0]       1
1        1  (-4.996190092599261, 5.241993379124888)       [0]       1
2        2  (-4.994066943961042, 5.245221042920774)       [1]       1
3        3  (-4.994633314086607, 5.242465375710156)       [1]       1
4        4   (-4.99443589237601, 5.243026440170422)       [2]       1

📋 Aperçu des angles calculés (5 premiers) :
   node_id                                   coords  angle      type  degree  \
0        0  (-4.997630958956445, 5.245875938860019)    NaN  terminal       1   
1        1  (-4.996190092599261, 5.241993379124888)    NaN  terminal       1   
2        2  (-4.994066943961042, 5.24522104292077

In [23]:
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point, LineString
from collections import defaultdict
from math import atan2, degrees
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
import warnings
warnings.filterwarnings('ignore')

# =============================================
# 1. CHARGEMENT DES DONNÉES
# =============================================

def load_data(filepath):
    """Charge les données GeoJSON des lignes BT"""
    try:
        gdf = gpd.read_file(filepath)
        if gdf.crs is None:
            gdf = gdf.set_crs(epsg=4326)
        print(f"✅ Données chargées: {len(gdf)} lignes")
        print(f"   CRS: {gdf.crs}")
        print(f"   Colonnes: {list(gdf.columns)}")
        return gdf
    except Exception as e:
        print(f"❌ Erreur de chargement: {e}")
        return None

# =============================================
# 2. EXTRACTION DES NŒUDS
# =============================================

def extract_nodes(gdf_lignes):
    """Extrait les nœuds à partir des extrémités des lignes"""
    
    node_connections = defaultdict(list)
    node_ids = {}
    node_coords_list = []

    for idx, row in gdf_lignes.iterrows():
        try:
            coords = list(row.geometry.coords)
        except:
            continue

        for point in [coords[0], coords[-1]]:
            # Arrondir pour éviter les doublons de précision flottante
            rounded = (round(point[0], 8), round(point[1], 8))
            if rounded not in node_ids:
                node_ids[rounded] = len(node_ids)
                node_coords_list.append(rounded)
            node_connections[node_ids[rounded]].append(idx)

    return node_ids, node_coords_list, node_connections

# =============================================
# 3. CALCUL DE L'ANGLE ENTRE DEUX VECTEURS
# =============================================

def calculate_angle(v1, v2):
    """Calcule l'angle en degrés entre deux vecteurs"""
    dot = v1[0]*v2[0] + v1[1]*v2[1]
    mag1 = sqrt(v1[0]**2 + v1[1]**2)
    mag2 = sqrt(v2[0]**2 + v2[1]**2)
    if mag1 == 0 or mag2 == 0:
        return 0.0
    cos_angle = max(-1.0, min(1.0, dot / (mag1 * mag2)))
    return degrees(np.arccos(cos_angle))

from math import sqrt

# =============================================
# 4. CLASSIFICATION DES SUPPORTS
# =============================================

def classify_nodes(gdf_lignes, node_ids, node_coords_list, node_connections, angle_threshold=15.0):
    """
    Classifie chaque nœud selon son type de support :
    - Arrêt      : degree = 1
    - Alignement : degree = 2, angle < seuil
    - Angle      : degree = 2, angle >= seuil
    - Dérivation : degree = 3
    - Étoilement : degree >= 4
    """

    records = []

    for coords, nid in node_ids.items():
        connected_lines = node_connections[nid]
        degree = len(connected_lines)
        angle_deg = None
        type_support = None

        # ── Degree 1 : ARRÊT ──────────────────────────────────────
        if degree == 1:
            type_support = "Arrêt"

        # ── Degree 2 : ALIGNEMENT ou ANGLE ────────────────────────
        elif degree == 2:
            # Récupérer les vecteurs directeurs depuis le nœud courant
            vectors = []
            for lid in connected_lines:
                try:
                    line_coords = list(gdf_lignes.loc[lid].geometry.coords)
                    start = (round(line_coords[0][0], 8), round(line_coords[0][1], 8))
                    end   = (round(line_coords[-1][0], 8), round(line_coords[-1][1], 8))

                    if start == coords:
                        # Le nœud est au début → vecteur vers la fin
                        vx = line_coords[1][0] - line_coords[0][0]
                        vy = line_coords[1][1] - line_coords[0][1]
                    else:
                        # Le nœud est à la fin → vecteur vers le début
                        n = len(line_coords)
                        vx = line_coords[n-2][0] - line_coords[n-1][0]
                        vy = line_coords[n-2][1] - line_coords[n-1][1]
                    vectors.append((vx, vy))
                except:
                    continue

            if len(vectors) == 2:
                angle_deg = calculate_angle(vectors[0], vectors[1])
                # L'angle de déviation = 180° - angle entre vecteurs sortants
                deviation = abs(180.0 - angle_deg)
                type_support = "Angle" if deviation >= angle_threshold else "Alignement"
            else:
                type_support = "Alignement"

        # ── Degree 3 : DÉRIVATION ──────────────────────────────────
        elif degree == 3:
            type_support = "Dérivation"

        # ── Degree >= 4 : ÉTOILEMENT ───────────────────────────────
        else:
            type_support = "Étoilement"

        records.append({
            'node_id'     : nid,
            'longitude'   : coords[0],
            'latitude'    : coords[1],
            'degree'      : degree,
            'angle_deg'   : round(angle_deg, 2) if angle_deg is not None else None,
            'type_support': type_support,
            'geometry'    : Point(coords[0], coords[1])
        })

    # Construire le GeoDataFrame
    df = pd.DataFrame(records)
    gdf_nodes = gpd.GeoDataFrame(df, geometry='geometry', crs=gdf_lignes.crs)

    return gdf_nodes

# =============================================
# 5. VISUALISATION
# =============================================

def visualize_network(gdf_lignes, gdf_nodes):
    """Visualise le réseau avec les nœuds classifiés et une légende détaillée"""

    # ── Configuration des styles par type ─────────────────────────
    style = {
        "Arrêt"      : {"color": "#E74C3C", "marker": "s", "size": 80,  "zorder": 5, "label": "Arrêt (degree=1)"},
        "Alignement" : {"color": "#2ECC71", "marker": "o", "size": 50,  "zorder": 4, "label": "Alignement (degree=2, angle<15°)"},
        "Angle"      : {"color": "#F39C12", "marker": "^", "size": 80,  "zorder": 5, "label": "Angle (degree=2, angle≥15°)"},
        "Dérivation" : {"color": "#3498DB", "marker": "D", "size": 90,  "zorder": 6, "label": "Dérivation (degree=3)"},
        "Étoilement" : {"color": "#9B59B6", "marker": "*", "size": 150, "zorder": 7, "label": "Étoilement (degree≥4)"},
    }

    fig, ax = plt.subplots(figsize=(16, 12))
    fig.patch.set_facecolor('#1a1a2e')
    ax.set_facecolor('#16213e')

    # ── Tracé des lignes ──────────────────────────────────────────
    gdf_lignes.plot(ax=ax, color='#778899', linewidth=0.8, alpha=0.7, zorder=2)

    # ── Tracé des nœuds par type ──────────────────────────────────
    legend_elements = []
    counts = gdf_nodes['type_support'].value_counts()

    for type_name, st in style.items():
        subset = gdf_nodes[gdf_nodes['type_support'] == type_name]
        if len(subset) == 0:
            continue

        ax.scatter(
            subset['longitude'], subset['latitude'],
            c=st['color'], marker=st['marker'],
            s=st['size'], zorder=st['zorder'],
            edgecolors='white', linewidths=0.4, alpha=0.95
        )

        count = counts.get(type_name, 0)
        legend_elements.append(
            Line2D([0], [0],
                   marker=st['marker'], color='w',
                   markerfacecolor=st['color'],
                   markersize=10,
                   label=f"{st['label']}  →  {count} nœuds",
                   linestyle='None')
        )

    # ── Légende ───────────────────────────────────────────────────
    legend = ax.legend(
        handles=legend_elements,
        loc='lower left',
        fontsize=9,
        title='🔌 Types de Supports BT',
        title_fontsize=10,
        facecolor='#0f3460',
        edgecolor='#e94560',
        labelcolor='white',
        framealpha=0.92,
        borderpad=1.0,
        labelspacing=0.8
    )
    legend.get_title().set_color('white')

    # ── Encadré de statistiques ───────────────────────────────────
    total = len(gdf_nodes)
    stats_lines = [f"📊 STATISTIQUES ({total} nœuds)"]
    stats_lines.append("─" * 30)
    for t, st in style.items():
        n = counts.get(t, 0)
        pct = n / total * 100 if total > 0 else 0
        stats_lines.append(f"  {t:<14}: {n:>4}  ({pct:5.1f}%)")

    # Angle moyen pour les nœuds degree=2
    angle_data = gdf_nodes[gdf_nodes['angle_deg'].notna()]
    if len(angle_data) > 0:
        stats_lines.append("─" * 30)
        stats_lines.append(f"  Angle moyen  : {angle_data['angle_deg'].mean():.1f}°")
        stats_lines.append(f"  Angle max    : {angle_data['angle_deg'].max():.1f}°")

    stats_text = "\n".join(stats_lines)
    ax.text(
        0.99, 0.99, stats_text,
        transform=ax.transAxes,
        fontsize=8.5,
        verticalalignment='top',
        horizontalalignment='right',
        bbox=dict(boxstyle='round,pad=0.6', facecolor='#0f3460',
                  edgecolor='#e94560', alpha=0.92),
        color='white',
        fontfamily='monospace'
    )

    # ── Titres et axes ────────────────────────────────────────────
    ax.set_title(
        "Réseau Basse Tension — Classification des Supports\n"
        "Arrêt | Alignement | Angle | Dérivation (3 lignes) | Étoilement (≥4 lignes)",
        fontsize=13, color='white', pad=15, fontweight='bold'
    )
    ax.set_xlabel("Longitude", color='#aaaaaa', fontsize=9)
    ax.set_ylabel("Latitude",  color='#aaaaaa', fontsize=9)
    ax.tick_params(colors='#aaaaaa', labelsize=8)
    for spine in ax.spines.values():
        spine.set_edgecolor('#444466')

    plt.tight_layout()
    output_path = "visualisation_supports_bt.png"
    plt.savefig(output_path, dpi=180, bbox_inches='tight',
                facecolor=fig.get_facecolor())
    plt.show()
    print(f"\n💾 Carte sauvegardée : {output_path}")

# =============================================
# 6. PROGRAMME PRINCIPAL
# =============================================

if __name__ == "__main__":

    FILEPATH = "LigneBT_GrandLahou_Ext_Dens_geojson.geojson"   # ← adapter le chemin

    try:
        # Chargement
        gdf_lignes = load_data(FILEPATH)
        if gdf_lignes is None:
            raise ValueError("Impossible de charger le fichier.")

        # Extraction des nœuds
        node_ids, node_coords_list, node_connections = extract_nodes(gdf_lignes)
        print(f"✅ {len(node_ids)} nœuds extraits")

        # Classification
        gdf_nodes = classify_nodes(
            gdf_lignes, node_ids, node_coords_list,
            node_connections, angle_threshold=15.0
        )

        # Affichage console
        print("\n📋 Répartition des types de supports :")
        print(gdf_nodes['type_support'].value_counts().to_string())

        print("\n📄 Aperçu des données :")
        print(gdf_nodes[['node_id','longitude','latitude',
                          'degree','angle_deg','type_support']].head(10).to_string(index=False))

        # Export CSV
        gdf_nodes.drop(columns='geometry').to_csv('supports_classifies.csv', index=False)
        print("\n💾 CSV exporté : supports_classifies.csv")

        # Visualisation
        visualize_network(gdf_lignes, gdf_nodes)

    except Exception as e:
        print(f"\n❌ Erreur : {e}")
        import traceback
        traceback.print_exc()


✅ Données chargées: 120 lignes
   CRS: EPSG:4326
   Colonnes: ['id', 'TFO', 'Sortie', 'Section', 'Ville', 'Longueur', 'Type', 'Type_2', 'geometry']
✅ 0 nœuds extraits

❌ Erreur : Unknown column geometry


Traceback (most recent call last):
  File "d:\Users\LENOVO\anaconda3\Lib\site-packages\geopandas\geodataframe.py", line 420, in set_geometry
    level = frame[col]
            ~~~~~^^^^^
  File "d:\Users\LENOVO\anaconda3\Lib\site-packages\geopandas\geodataframe.py", line 1891, in __getitem__
    result = super().__getitem__(key)
             ^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\Users\LENOVO\anaconda3\Lib\site-packages\pandas\core\frame.py", line 4378, in __getitem__
    indexer = self.columns.get_loc(key)
              ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\Users\LENOVO\anaconda3\Lib\site-packages\pandas\core\indexes\range.py", line 525, in get_loc
    raise KeyError(key)
KeyError: 'geometry'

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "C:\Users\LENOVO\AppData\Local\Temp\ipykernel_30828\1068634832.py", line 287, in <module>
    gdf_nodes = classify_nodes(
                ^^^^^^^^^^^^^^^
  File "C:\Users\LENOVO\AppData\Loc